In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Obtain information from the database using SQL queries"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [4]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools)

    return handler(request)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

275 artists.

Note: The correct table name is Artist (capital A). The query SELECT COUNT(*) FROM Artist returned 275. If you’d like, I can list the artist names or show additional stats (e.g., top genres, distribution by country).


In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database. Tell me which database or dataset you’re referring to, and whether you want the total count of artists or the count of distinct artists (e.g., by artist_id). If you share the schema, I can tailor the query. In the meantime, here are common options:

SQL (PostgreSQL/MySQL/SQLite)
- Total artists in table:
  SELECT COUNT(*) AS artist_count FROM artists;

- Distinct artists by id (in case of duplicates):
  SELECT COUNT(DISTINCT artist_id) AS artist_count FROM artists;

- Only artists with at least one artwork (assuming artworks table with artist_id):
  SELECT COUNT(DISTINCT a.artist_id) AS artist_count
  FROM artists a
  JOIN artworks w ON a.artist_id = w.artist_id;

MongoDB
- Total documents in the artists collection:
  db.artists.countDocuments({})

- Approximate (large collections):
  db.artists.estimatedDocumentCount()

- Distinct by _id (equivalent to distinct artists if each id is unique):
  db.artists.distinct("_id").length

If you share your d